<a href="https://colab.research.google.com/github/arun-antony/ai-colab-tryout/blob/main/capstone_personalised_recommendation_retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries (as suggested in generic notebook)
!pip install -q sentence-transformers faiss-cpu pandas numpy

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load datasets
products = pd.read_csv('retail_products.csv')
customers = pd.read_csv('retail_customers.csv')
events = pd.read_csv('retail_events.csv')

In [ ]:
# Create a text representation for each product, this will be used for creating the embedding
products['combined_text'] = products.apply(
    lambda row: f"Product: {row['category']} by {row['brand']}", axis=1
)

print("Sample Product Description:")
print(products['combined_text'].iloc[0])

In [ ]:
# Load the model from generic notebook
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all products
product_embeddings = embedding_model.encode(products['combined_text'].tolist(), show_progress_bar=True)

# Convert to float32 for FAISS compatibility
product_embeddings = np.array(product_embeddings).astype('float32')

In [ ]:
# Initialize FAISS index
dimension = product_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(product_embeddings)

print(f"Indexed {index.ntotal} products in vector store.")

In [ ]:
def recommend_by_category(customer_id, k_per_category=3):
    # 1. Get user events
    user_events = events[events['customer_id'] == customer_id].copy()

    final_recommendations = {}
    seen_product_ids = set()

    # We prioritize categories: Purchase > Cart > View
    categories = ['purchase', 'cart', 'view']

    for cat in categories:
        cat_events = user_events[user_events['event_type'] == cat]

        if cat_events.empty:
            continue

        # 2. Create a "Category Vector" (Mean of product embeddings in this specific category)

        # Convert Product IDs (text) into Row Indices (integers) so we can look up their vectors in the embedding matrix
        interacted_indices = products[products['product_id'].isin(cat_events['product_id'])].index.tolist()

        # now create a list of embeddings that the user interacted with for the given category
        cat_embeddings = product_embeddings[interacted_indices]

        # Create a single 'Average Taste Vector' by calculating the mean of all product embeddings in this category;
        # Reshape to 2D and convert to float32 to meet the strict technical requirements of the FAISS vector engine.
        query_vector = np.mean(cat_embeddings, axis=0).reshape(1, -1).astype('float32')

        # 3. Search FAISS (requesting more than k to account for deduplication)
        distances, indices = index.search(query_vector, k_per_category + 5)

        # 4. Filter for uniqueness and relevance
        cat_results = []
        for idx in indices[0]:
            pid = products.iloc[idx]['product_id']
            # Only add if it's unique AND the user hasn't already interacted with it
            if pid not in seen_product_ids and pid not in user_events['product_id'].values:
                cat_results.append(products.iloc[idx])
                seen_product_ids.add(pid)

            if len(cat_results) >= k_per_category:
                break

        final_recommendations[cat] = pd.DataFrame(cat_results)

    return final_recommendations

In [ ]:
# To authenticate with Hugging Face and access gated models, you need to log in.
# 1. Go to https://huggingface.co/settings/tokens to create a new token.
# 2. In Google Colab, go to the left panel (the key icon) to 'Secrets'.
# 3. Add a new secret with the name 'HF_TOKEN' and paste your Hugging Face token as the value.
# 4. Then, run the following code:

from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab secrets
hf_token = userdata.get('hf-colab')

# Log in to Hugging Face
login(token=hf_token)

print("Successfully logged in to Hugging Face.")

In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

model_id = "google/gemma-3-4b-it"

llm = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
def generate_explanation_with_gemma(customer_id, interaction_type, rec_df):
    # --- Data Preparation ---
    past_interactions = events[(events['customer_id'] == customer_id) &
                               (events['event_type'] == interaction_type)]
    past_products = products[products['product_id'].isin(past_interactions['product_id'])]

    history_str = ", ".join([f"{r.brand} {r.category}" for _, r in past_products.iterrows()])
    rec_str = ", ".join([f"{r.brand} {r.category}" for _, r in rec_df.iterrows()])

    print(f"{interaction_type} History: {history_str}")
    print(f"Recommendations: {rec_str}")
    # --- Gemma 3 Formatting ---
    messages = [
        {"role": "system", "content": f"You are a helpful retail assistant. Explain recommendations clearly based on {interaction_type} history."},
        {"role": "user", "content": f"The customer has a history of {interaction_type}s: {history_str}. Explain why they might like these new items: {rec_str}."}
    ]

    # Process the prompt
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt, return_tensors="pt").to(llm.device)

    # Generate
    output_tokens = llm.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.1
    )

    # Decode only the new generated text
    new_tokens = output_tokens[0][len(inputs.input_ids[0]):]
    return processor.decode(new_tokens, skip_special_tokens=True)

In [ ]:
test_rec = recommend_by_category("C000467")
print(test_rec["view"])

# for cat, df in test_rec.items():
#     print(f"\n--- Recommendations based on your {cat.upper()}S ---")
#     if not df.empty:
#         display(df[['product_id', 'category', 'brand', 'price']])
#     else:
#         print("No recommendations for this category.")

In [ ]:
result = generate_explanation_with_gemma("C000467", "cart", test_rec["cart"])
print(result)